# Análisis exploratorio BTS

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("eda_bts_on_time")
    .enableHiveSupport()
    .getOrCreate()
)
spark.conf.set("spark.sql.shuffle.partitions", "48")

RAW_BASE = "/Obligatorio/landing/bts"

# Detectamos las carpetas year=YYYY presentes en HDFS (esperado: 2023, 2024, 2025)
sc = spark.sparkContext
hadoop = sc._jvm.org.apache.hadoop.fs
fs = hadoop.FileSystem.get(sc._jsc.hadoopConfiguration())
carpetas = sorted(
    s.getPath().toString() for s in fs.listStatus(hadoop.Path(RAW_BASE))
    if s.getPath().getName().startswith("year=")
)
print("Carpetas BTS encontradas:")
for c in carpetas: print(" ", c)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-20T20:59:39,202 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Carpetas BTS encontradas:
  hdfs://localhost:9000/Obligatorio/landing/bts/year=2023
  hdfs://localhost:9000/Obligatorio/landing/bts/year=2024
  hdfs://localhost:9000/Obligatorio/landing/bts/year=2025


In [3]:
# Lectura recursiva de todos los CSV bajo /Obligatorio/raw/bts (todos los años/meses).
# Leemos como texto (sin inferSchema) por volumen y para controlar el tipado nosotros.
bts_raw = (
    spark.read
    .option("header", True)
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.csv")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(RAW_BASE)
)

print("Filas totales (todos los años/meses):", bts_raw.count())
print("Columnas leídas:", len(bts_raw.columns))



[Stage 2:========================================================>(73 + 1) / 74]

Filas totales (todos los años/meses): 20928579
Columnas leídas: 110


In [4]:
# El header de BTS termina en coma -> aparece una columna vacía al final.
# La identificamos para descartarla.
print("Últimas columnas leídas:", bts_raw.columns[-3:])

fantasma = [c for c in bts_raw.columns if c.strip() == "" or c.startswith("_c")]
print("Columnas fantasma a descartar:", fantasma)

if fantasma:
    bts_raw = bts_raw.drop(*fantasma)
print("Columnas tras quitar fantasma:", len(bts_raw.columns))

# Verificamos que la lectura unificada respeta una sola estructura de columnas
# (si algún archivo difiriera, lo veríamos en el conteo de columnas o en columnas extra).
print("Nombres de columnas:")
print(bts_raw.columns)



Últimas columnas leídas: ['Div5WheelsOff', 'Div5TailNum', '_c109']
Columnas fantasma a descartar: ['_c109']
Columnas tras quitar fantasma: 109
Nombres de columnas:
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 

In [5]:
(bts_raw
    .groupBy("Year", "Month").count()
    .orderBy("Year", "Month")
    .show(40, truncate=False))

print("Años presentes:")
bts_raw.select("Year").distinct().orderBy("Year").show()



+----+-----+------+
|Year|Month|count |
+----+-----+------+
|2023|1    |538837|
|2023|10   |598968|
|2023|11   |563777|
|2023|12   |570394|
|2023|2    |502749|
|2023|3    |580322|
|2023|4    |561441|
|2023|5    |579958|
|2023|6    |577262|
|2023|7    |601866|
|2023|8    |602987|
|2023|9    |569338|
|2024|1    |547271|
|2024|10   |615497|
|2024|11   |575404|
|2024|12   |590581|
|2024|2    |519221|
|2024|3    |591767|
|2024|4    |582185|
|2024|5    |609743|
|2024|6    |611132|
|2024|7    |634613|
|2024|8    |619025|
|2024|9    |582622|
|2025|1    |539747|
|2025|10   |605844|
|2025|11   |570550|
|2025|12   |582304|
|2025|2    |504884|
|2025|3    |600872|
|2025|4    |583950|
|2025|5    |605648|
|2025|6    |611575|
|2025|7    |631428|
|2025|8    |602378|
|2025|9    |562439|
+----+-----+------+

Años presentes:


[Stage 8:========================================================>(73 + 1) / 74]

+----+
|Year|
+----+
|2023|
|2024|
|2025|
+----+



In [6]:
# Tomamos una muestra para no escanear millones de filas en 100+ columnas.
muestra = bts_raw.sample(fraction=0.02, seed=42).cache()
n = muestra.count()
print("Filas en la muestra:", n)

# Bloques candidatos a eliminar (desvíos y metadatos no usados)
cols_descartables = [c for c in bts_raw.columns if c.startswith("Div")] + [
    "OriginAirportSeqID","OriginCityMarketID","OriginStateFips","OriginWac",
    "DestAirportSeqID","DestCityMarketID","DestStateFips","DestWac",
    "DepartureDelayGroups","ArrivalDelayGroups","DepTimeBlk","ArrTimeBlk",
    "WheelsOff","WheelsOn","FirstDepTime","TotalAddGTime","LongestAddGTime",
    "DistanceGroup","Tail_Number"
]
cols_descartables = [c for c in cols_descartables if c in muestra.columns]

filas = []
for c in cols_descartables:
    col_str = F.trim(F.col(c).cast("string"))
    k = muestra.filter(F.col(c).isNull() | col_str.isin(["", "\\N"])).count()
    filas.append((c, round(100*k/n, 1)))
print("\n% de nulos (muestra) en columnas candidatas a descartar:")
(spark.createDataFrame(filas, ["columna", "pct_nulos"])
      .orderBy(F.desc("pct_nulos")).show(100, truncate=False))
muestra.unpersist()


2026-06-20T21:05:56,592 WARN [Thread-4] org.apache.spark.sql.catalyst.util.package - Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Filas en la muestra: 418010



% de nulos (muestra) en columnas candidatas a descartar:


[Stage 209:>                                                        (0 + 2) / 2]

+--------------------+---------+
|columna             |pct_nulos|
+--------------------+---------+
|Div2Airport         |100.0    |
|Div4AirportSeqID    |100.0    |
|Div2AirportID       |100.0    |
|Div4WheelsOn        |100.0    |
|Div2AirportSeqID    |100.0    |
|Div4TotalGTime      |100.0    |
|Div2WheelsOn        |100.0    |
|Div4LongestGTime    |100.0    |
|Div2TotalGTime      |100.0    |
|Div4WheelsOff       |100.0    |
|Div2LongestGTime    |100.0    |
|Div4TailNum         |100.0    |
|Div2WheelsOff       |100.0    |
|Div5Airport         |100.0    |
|Div2TailNum         |100.0    |
|Div5AirportID       |100.0    |
|Div3Airport         |100.0    |
|Div5AirportSeqID    |100.0    |
|Div3AirportID       |100.0    |
|Div5WheelsOn        |100.0    |
|Div3AirportSeqID    |100.0    |
|Div5TotalGTime      |100.0    |
|Div3WheelsOn        |100.0    |
|Div5LongestGTime    |100.0    |
|Div3TotalGTime      |100.0    |
|Div5WheelsOff       |100.0    |
|Div3LongestGTime    |100.0    |
|Div5TailN

DataFrame[Year: string, Quarter: string, Month: string, DayofMonth: string, DayOfWeek: string, FlightDate: string, Reporting_Airline: string, DOT_ID_Reporting_Airline: string, IATA_CODE_Reporting_Airline: string, Tail_Number: string, Flight_Number_Reporting_Airline: string, OriginAirportID: string, OriginAirportSeqID: string, OriginCityMarketID: string, Origin: string, OriginCityName: string, OriginState: string, OriginStateFips: string, OriginStateName: string, OriginWac: string, DestAirportID: string, DestAirportSeqID: string, DestCityMarketID: string, Dest: string, DestCityName: string, DestState: string, DestStateFips: string, DestStateName: string, DestWac: string, CRSDepTime: string, DepTime: string, DepDelay: string, DepDelayMinutes: string, DepDel15: string, DepartureDelayGroups: string, DepTimeBlk: string, TaxiOut: string, WheelsOff: string, WheelsOn: string, TaxiIn: string, CRSArrTime: string, ArrTime: string, ArrDelay: string, ArrDelayMinutes: string, ArrDel15: string, Arriv

In [8]:
def limpiar(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s.isin(["", "\\N"]), None).otherwise(s)

bts = bts_raw.select(
    F.col("Year").cast("int").alias("year"),
    F.col("Quarter").cast("int").alias("quarter"),
    F.col("Month").cast("int").alias("month"),
    F.col("DayofMonth").cast("int").alias("day_of_month"),
    F.col("DayOfWeek").cast("int").alias("day_of_week"),
    F.to_date(limpiar("FlightDate")).alias("flight_date"),
    F.upper(limpiar("Reporting_Airline")).alias("reporting_airline"),
    F.upper(limpiar("IATA_CODE_Reporting_Airline")).alias("iata_airline_code"),
    F.col("OriginAirportID").cast("int").alias("origin_airport_id"),
    F.upper(limpiar("Origin")).alias("origin"),
    limpiar("OriginCityName").alias("origin_city_name"),
    F.upper(limpiar("OriginState")).alias("origin_state"),
    F.col("DestAirportID").cast("int").alias("dest_airport_id"),
    F.upper(limpiar("Dest")).alias("dest"),
    limpiar("DestCityName").alias("dest_city_name"),
    F.upper(limpiar("DestState")).alias("dest_state"),
    F.col("DepDelay").cast("double").alias("dep_delay"),
    F.col("DepDelayMinutes").cast("double").alias("dep_delay_minutes"),
    F.col("DepDel15").cast("double").alias("dep_del15"),
    F.col("TaxiOut").cast("double").alias("taxi_out"),
    F.col("ArrDelay").cast("double").alias("arr_delay"),
    F.col("ArrDelayMinutes").cast("double").alias("arr_delay_minutes"),
    F.col("ArrDel15").cast("double").alias("arr_del15"),
    F.col("TaxiIn").cast("double").alias("taxi_in"),
    F.col("Cancelled").cast("double").alias("cancelled"),
    limpiar("CancellationCode").alias("cancellation_code"),
    F.col("Diverted").cast("double").alias("diverted"),
    F.col("Distance").cast("double").alias("distance"),
    F.col("Flights").cast("double").alias("flights"),
    F.col("CarrierDelay").cast("double").alias("carrier_delay"),
    F.col("WeatherDelay").cast("double").alias("weather_delay"),
    F.col("NASDelay").cast("double").alias("nas_delay"),
    F.col("SecurityDelay").cast("double").alias("security_delay"),
    F.col("LateAircraftDelay").cast("double").alias("late_aircraft_delay"),
)
bts.printSchema()
print("Columnas conservadas:", len(bts.columns))



root
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- reporting_airline: string (nullable = true)
 |-- iata_airline_code: string (nullable = true)
 |-- origin_airport_id: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- origin_city_name: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- dest_airport_id: integer (nullable = true)
 |-- dest: string (nullable = true)
 |-- dest_city_name: string (nullable = true)
 |-- dest_state: string (nullable = true)
 |-- dep_delay: double (nullable = true)
 |-- dep_delay_minutes: double (nullable = true)
 |-- dep_del15: double (nullable = true)
 |-- taxi_out: double (nullable = true)
 |-- arr_delay: double (nullable = true)
 |-- arr_delay_minutes: double (nullable = true)
 |-- arr_del15: double (nullable = true

In [9]:
# Para no recorrer la tabla una vez por columna, agregamos todo en una sola pasada.
exprs = [
    F.sum(
        (F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")).cast("int")
    ).alias(c)
    for c in bts.columns
]
total = bts.count()
nulos = bts.select(*exprs).collect()[0].asDict()
filas = [(c, int(v), round(100*v/total, 1)) for c, v in nulos.items()]
print(f"Total filas: {total}")
(spark.createDataFrame(filas, ["columna", "nulos", "porcentaje"])
      .orderBy(F.desc("nulos")).show(100, truncate=False))



Total filas: 20928579
+-------------------+--------+----------+
|columna            |nulos   |porcentaje|
+-------------------+--------+----------+
|cancellation_code  |20641445|98.6      |
|carrier_delay      |16557279|79.1      |
|weather_delay      |16557279|79.1      |
|nas_delay          |16557279|79.1      |
|security_delay     |16557279|79.1      |
|late_aircraft_delay|16557279|79.1      |
|arr_delay          |340445  |1.6       |
|arr_delay_minutes  |340445  |1.6       |
|arr_del15          |340445  |1.6       |
|taxi_in            |292087  |1.4       |
|taxi_out           |285189  |1.4       |
|dep_delay          |276090  |1.3       |
|dep_delay_minutes  |276090  |1.3       |
|dep_del15          |276090  |1.3       |
|year               |0       |0.0       |
|cancelled          |0       |0.0       |
|quarter            |0       |0.0       |
|diverted           |0       |0.0       |
|month              |0       |0.0       |
|distance           |0       |0.0       |
|day_of_mont

In [10]:
# La granularidad es el vuelo individual; no hay un id único de fila.
# Una clave de negocio razonable para detectar duplicados de vuelo:
clave_vuelo = ["flight_date", "reporting_airline",
               "origin_airport_id", "dest_airport_id", "dep_delay", "taxi_out"]

total = bts.count()
print("Filas:", total)
print("Filas exactas duplicadas:", total - bts.dropDuplicates().count())
print("Duplicados por clave de vuelo aproximada:",
      total - bts.dropDuplicates(clave_vuelo).count())

# Campos mínimos que NO pueden faltar para el análisis operativo
faltan_minimos = bts.filter(
    F.col("year").isNull() | F.col("month").isNull() | F.col("flight_date").isNull() |
    F.col("origin").isNull() | F.col("dest").isNull() | F.col("reporting_airline").isNull()
).count()
print("Filas sin campos mínimos (a descartar):", faltan_minimos)



Filas: 20928579


Filas exactas duplicadas: 53885


[Stage 427:===============================================>         (5 + 1) / 6]

Duplicados por clave de vuelo aproximada: 164968


[Stage 431:======================================================>(73 + 1) / 74]

Filas sin campos mínimos (a descartar): 0


In [11]:
bts.select(
    F.min("flight_date").alias("fecha_min"),
    F.max("flight_date").alias("fecha_max"),
    F.min("year").alias("anio_min"),
    F.max("year").alias("anio_max"),
    F.countDistinct("origin").alias("aeropuertos_origen"),
    F.countDistinct("dest").alias("aeropuertos_destino"),
    F.countDistinct("reporting_airline").alias("aerolineas")
).show(truncate=False)

# Años fuera del período esperado (deberían ser 0 si solo hay 2023-2025)
print("Filas con año < 2023:", bts.filter(F.col("year") < 2023).count())


+----------+----------+--------+--------+------------------+-------------------+----------+
|fecha_min |fecha_max |anio_min|anio_max|aeropuertos_origen|aeropuertos_destino|aerolineas|
+----------+----------+--------+--------+------------------+-------------------+----------+
|2023-01-01|2025-12-31|2023    |2025    |362               |362                |15        |
+----------+----------+--------+--------+------------------+-------------------+----------+



[Stage 440:======================================================>(73 + 1) / 74]

Filas con año < 2023: 0


In [12]:
# Cancelled / Diverted: deberían ser 0/1. Justifica tratarlos como indicadores.
print("Distribución de 'cancelled':")
bts.groupBy("cancelled").count().orderBy("cancelled").show()
print("Distribución de 'diverted':")
bts.groupBy("diverted").count().orderBy("diverted").show()

# Las causas de demora están vacías cuando el vuelo no tuvo demora atribuida:
# esto es un NULO LEGÍTIMO, no un error de calidad. Lo mostramos para documentarlo.
total = bts.count()
con_causa = bts.filter(F.col("carrier_delay").isNotNull()).count()
print(f"\nVuelos con causa de demora informada: {con_causa} de {total} "
      f"({round(100*con_causa/total,1)}%) -> el resto son nulos esperables")


Distribución de 'cancelled':


+---------+--------+
|cancelled|   count|
+---------+--------+
|      0.0|20641445|
|      1.0|  287134|
+---------+--------+

Distribución de 'diverted':


+--------+--------+
|diverted|   count|
+--------+--------+
|     0.0|20875270|
|     1.0|   53309|
+--------+--------+



[Stage 452:======================================================>(73 + 1) / 74]


Vuelos con causa de demora informada: 4371300 de 20928579 (20.9%) -> el resto son nulos esperables
